# 第8章　実践②　カラー画像分類（CIFAR-10）＋ データ拡張

第6章の MNIST（白黒・簡単）の次のステップ。**CIFAR-10** は 32×32 の**カラー写真**10種類
（飛行機・車・鳥・猫・鹿・犬・カエル・馬・船・トラック）。MNIST よりずっと難しく、
**カラー（3チャンネル）・データ拡張・深いCNN・過学習対策**を学ぶのに最適です。

ゴール：カラー画像CNN を組み、データ拡張と Dropout/BatchNorm で精度を上げる感覚をつかむ。

> **使い方**：上から順に `Shift + Enter`。GPU 推奨の章です（Colab：ランタイム→ランタイムのタイプを変更→GPU）。

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 8-1. MNIST との違い

| | MNIST | CIFAR-10 |
|---|---|---|
| 画像 | 白黒 28×28 | **カラー 32×32** |
| 形 | `(N, 1, 28, 28)` | **`(N, 3, 32, 32)`**（C=3=RGB） |
| 難易度 | 簡単（99%可） | 難しい（簡単なCNNで 75〜82%） |

ポイント：入力チャンネルが **1→3** に増えるので、最初の `Conv2d` の `in_channels=3` にします。

## 8-2. データ拡張（Data Augmentation）

学習画像を**ランダムに少し変形**（切り出し・左右反転など）して水増しすると、
モデルが「位置や向きが多少違っても同じ猫」と学べて**過学習しにくく**なります。

- **学習用だけ**に拡張をかける。**テスト用は素のまま**（評価は公平に）。
- `Normalize` の平均・標準偏差は CIFAR-10 の定番値を使用。

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),     # 4px余白を付けてランダム切り出し
    transforms.RandomHorizontalFlip(),        # 左右反転
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
test_ds  = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)
print("train:", len(train_ds), " test:", len(test_ds))

### 画像を見てみる（正規化を戻して表示）

In [ ]:
import matplotlib.pyplot as plt
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
m = torch.tensor(mean).view(3,1,1); s = torch.tensor(std).view(3,1,1)

xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, i in zip(axes, range(6)):
    img = (xb[i]*s + m).clamp(0,1).permute(1,2,0)   # 正規化を戻し (H,W,C) へ
    ax.imshow(img); ax.set_title(classes[yb[i]], fontsize=9); ax.axis("off")
plt.show()

## 8-3. 深めのCNN（BatchNorm + Dropout 入り）

- **`BatchNorm2d`**：各層の出力を整える → 学習が速く・安定する。
- **`Dropout`**：学習中にランダムにニューロンを切る → 過学習を防ぐ。
- Conv→BN→ReLU→Pool を3段重ねて、32×32 を 4×4 まで縮小。

In [ ]:
import torch.nn as nn

class CIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),  # 32->16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),  # 16->8
            nn.Conv2d(64, 128, 3, padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),  # 8->4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = CIFARNet().to(device)
# 形のチェック（ダミー入力 2枚）
dummy = torch.randn(2, 3, 32, 32).to(device)
print("出力 shape:", model(dummy).shape)   # (2, 10)

## 8-4. 学習（GPU 推奨）
第6章と同じ5ステップ。`EPOCHS` は GPU なら 10〜20 で 80%超。CPU なら 2〜3 に減らしてOK（精度は落ちます）。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"epoch {epoch+1:2d}: 平均loss = {running/len(train_loader):.4f}")

## 8-5. テスト精度

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f"テスト正解率: {100*correct/total:.2f}%")

## 8-6. 過学習（overfitting）の話

CIFAR-10 では「**学習データの精度は高いのにテストは伸びない**」が起きがち。これが過学習。
対策が今回入れた **データ拡張・Dropout・BatchNorm**。それでも簡単なCNNの限界は 80%台前半で、
さらに上げるには **ResNet** などの工夫（残差接続）や学習率スケジューリングが必要です（→ 演習）。

## 演習 8
1. データ拡張を**外して**（`train_tf` を `test_tf` と同じに）学習し、train とテストの精度差（過学習）が広がるのを観察しよう。
2. `EPOCHS` を増やし、学習率スケジューラ `torch.optim.lr_scheduler.CosineAnnealingLR` を足して精度向上を狙おう。
3. `Dropout` の割合（0.3→0.5）や `BatchNorm` の有無で挙動がどう変わるか比べよう。
4. 余裕があれば `torchvision.models.resnet18(num_classes=10)` に差し替えて比較（CIFAR用に1層目を調整すると尚良）。

In [ ]:
# ここに自分のコードを書いて実行してみよう
